# Week 2: MNIST Image Classifier with PyTorch

本 Notebook 是暑期 AI 工程实践训练营第二周完整实验。

你将完成：

1. 学习 Tensor 基础。
2. 加载 MNIST 或备用 digits 数据集。
3. 使用 Dataset 和 DataLoader 批量读取数据。
4. 构建 MLP 图像分类模型。
5. 训练模型并记录 Loss 和 Accuracy。
6. 保存模型、训练曲线、预测样例和评估指标。

请从上到下逐个 cell 运行。

## 1. 导入依赖并创建目录

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

try:
    from torchvision import datasets, transforms
    TORCHVISION_AVAILABLE = True
except Exception as exc:
    print("torchvision 不可用，将使用备用 digits 数据集。错误信息:", exc)
    TORCHVISION_AVAILABLE = False

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT_DIR / "data"
FIGURE_DIR = ROOT_DIR / "figures"
MODEL_DIR = ROOT_DIR / "models"
RESULT_DIR = ROOT_DIR / "results"

for directory in [DATA_DIR, FIGURE_DIR, MODEL_DIR, RESULT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("工程根目录:", ROOT_DIR)
print("PyTorch version:", torch.__version__)
print("torchvision available:", TORCHVISION_AVAILABLE)

## 2. 检查计算设备

如果电脑有可用 GPU，会使用 `cuda`；否则使用 `cpu`。没有 GPU 也可以完成本实验。

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("当前使用设备:", device)

## 3. Tensor 基础

Tensor 是 PyTorch 中最核心的数据结构，可以理解成支持 GPU 和自动求导的多维数组。

In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y = torch.ones_like(x)

print("x:")
print(x)
print("shape:", x.shape)
print("dtype:", x.dtype)
print("x + y:")
print(x + y)

## 4. 加载数据集

优先使用 MNIST 手写数字数据集。如果 MNIST 下载失败，会自动使用 scikit-learn 的 digits 数据集作为备用。

In [ ]:
BATCH_SIZE = 64

def load_mnist_or_digits():
    if TORCHVISION_AVAILABLE:
        try:
            transform = transforms.ToTensor()
            train_dataset = datasets.MNIST(
                root=str(DATA_DIR),
                train=True,
                download=True,
                transform=transform,
            )
            test_dataset = datasets.MNIST(
                root=str(DATA_DIR),
                train=False,
                download=True,
                transform=transform,
            )
            return train_dataset, test_dataset, "MNIST", 28, 28
        except Exception as exc:
            print("MNIST 下载失败，使用备用 digits 数据集。错误信息:", exc)

    digits = load_digits()
    X = digits.images.astype("float32") / 16.0
    y = digits.target.astype("int64")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    X_train = torch.tensor(X_train).unsqueeze(1)
    X_test = torch.tensor(X_test).unsqueeze(1)
    y_train = torch.tensor(y_train)
    y_test = torch.tensor(y_test)

    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)
    return train_dataset, test_dataset, "sklearn_digits", 8, 8


train_dataset, test_dataset, dataset_name, image_height, image_width = load_mnist_or_digits()

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("数据集:", dataset_name)
print("训练样本数:", len(train_dataset))
print("测试样本数:", len(test_dataset))
print("图像尺寸:", image_height, "x", image_width)

## 5. 查看一个 Batch

DataLoader 每次返回一批图片和一批标签。

In [ ]:
images, labels = next(iter(train_loader))

print("images shape:", images.shape)
print("labels shape:", labels.shape)
print("labels 前 10 个:", labels[:10].tolist())

shape 解释：

- `batch_size`：一批有多少张图。
- `channel`：灰度图是 1。
- `height` 和 `width`：图片高和宽。

## 6. 可视化样本图片

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i].squeeze().numpy(), cmap="gray")
    plt.title(f"label={labels[i].item()}")
    plt.axis("off")

plt.tight_layout()
sample_images_path = FIGURE_DIR / "sample_images.png"
plt.savefig(sample_images_path, dpi=150)
plt.show()

print("样本图片已保存:", sample_images_path)

## 7. 定义 MLP 模型

MLP 不能直接处理二维图片，所以先用 `Flatten` 把图片拉平成一维向量。

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, image_height, image_width, num_classes=10):
        super().__init__()
        input_dim = image_height * image_width
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)


model = MLPClassifier(image_height=image_height, image_width=image_width).to(device)
model

## 8. 定义 Loss 和 Optimizer

- Loss 衡量模型错得有多离谱。
- Optimizer 根据 Loss 更新模型参数。

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Loss:", criterion)
print("Optimizer:", optimizer.__class__.__name__)

## 9. 编写训练和评估函数

In [ ]:
def train_one_epoch(model, data_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

## 10. 训练模型

如果机器较慢，可以把 `EPOCHS` 改成 1。

In [ ]:
EPOCHS = 3

history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
    }
    history.append(row)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
        f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}"
    )

history_df = pd.DataFrame(history)
history_df

## 11. 保存训练指标

In [ ]:
metrics_path = RESULT_DIR / "metrics.csv"
history_df.to_csv(metrics_path, index=False)
print("训练指标已保存:", metrics_path)

## 12. 绘制训练曲线

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
plt.plot(history_df["epoch"], history_df["test_loss"], marker="o", label="test")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df["epoch"], history_df["train_acc"], marker="o", label="train")
plt.plot(history_df["epoch"], history_df["test_acc"], marker="o", label="test")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")
plt.legend()

plt.tight_layout()
training_curves_path = FIGURE_DIR / "training_curves.png"
plt.savefig(training_curves_path, dpi=150)
plt.show()

print("训练曲线已保存:", training_curves_path)

## 13. 保存模型

In [ ]:
model_path = MODEL_DIR / "mnist_mlp.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "image_height": image_height,
    "image_width": image_width,
    "dataset_name": dataset_name,
}, model_path)

print("模型已保存:", model_path)
print("模型文件是否存在:", model_path.exists())

## 14. 加载模型并预测样本

In [ ]:
checkpoint = torch.load(model_path, map_location=device)
loaded_model = MLPClassifier(
    image_height=checkpoint["image_height"],
    image_width=checkpoint["image_width"],
).to(device)
loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model.eval()

test_images, test_labels = next(iter(test_loader))
test_images = test_images.to(device)

with torch.no_grad():
    outputs = loaded_model(test_images)
    preds = outputs.argmax(dim=1).cpu()

print("真实标签:", test_labels[:10].tolist())
print("预测标签:", preds[:10].tolist())

## 15. 可视化预测结果

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(test_images[i].cpu().squeeze().numpy(), cmap="gray")
    plt.title(f"T={test_labels[i].item()}, P={preds[i].item()}")
    plt.axis("off")

plt.tight_layout()
prediction_examples_path = FIGURE_DIR / "prediction_examples.png"
plt.savefig(prediction_examples_path, dpi=150)
plt.show()

print("预测样例图已保存:", prediction_examples_path)

## 16. 最终检查

In [ ]:
expected_files = [
    FIGURE_DIR / "sample_images.png",
    FIGURE_DIR / "training_curves.png",
    FIGURE_DIR / "prediction_examples.png",
    MODEL_DIR / "mnist_mlp.pt",
    RESULT_DIR / "metrics.csv",
]

for path in expected_files:
    status = "OK" if path.exists() else "MISSING"
    print(f"{status}: {path}")

## 17. 学习复盘

运行完成后，请尝试回答：

1. Tensor 和 NumPy array 有什么相似和不同？
2. Dataset 和 DataLoader 分别解决什么问题？
3. 为什么训练时要使用 Batch？
4. MLP 为什么要先 Flatten 图片？
5. CrossEntropyLoss 适合什么任务？
6. Optimizer 在训练中做了什么？
7. train 模式和 eval 模式有什么区别？
8. CNN 相比 MLP 为什么更适合图像？